# Time Series Forecasting with LSTM — A Gentle Intro

## What it is
An LSTM is a recurrent neural network that learns temporal dependencies with gating (memory) mechanisms.

## What it does
- Consumes a sequence of past values and predicts the next value (sequence-to-one).
- Handles long- and short-term patterns via input/forget/output gates.

## When to use
- One-step or multi-step forecasting where history matters (seasonality/trends).

In [ ]:
import numpy as np  # numerical arrays
import torch  # PyTorch core
import torch.nn as nn  # neural network layers
import torch.optim as optim  # optimization algorithms
import matplotlib.pyplot as plt  # plotting

device = torch.device('cpu')  # use CPU for portability
np.random.seed(0)  # reproducibility for NumPy ops
torch.manual_seed(0)  # reproducibility for Torch ops

# Synthetic signal: trend + two seasonal components + noise
T = 500  # total time steps
t = np.arange(T)  # time index
series = 0.4*(t/T) + np.sin(2*np.pi*t/50) + 0.6*np.sin(2*np.pi*t/12) + np.random.normal(0, 0.08, T)  # target signal

In [ ]:
# Convert series into (window -> next value) supervised samples
def create_dataset(series, window=24):  # window: number of past steps
    X, y = [], []  # lists to collect samples
    for i in range(len(series) - window):  # slide over the series
        X.append(series[i:i+window])  # past window values
        y.append(series[i+window])  # next value (label)
    X = np.array(X)  # to NumPy array
    y = np.array(y)  # to NumPy array
    X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)  # shape (N, window, 1)
    y = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)  # shape (N, 1)
    return X, y  # tensors ready for training

WINDOW = 24  # past steps provided to the model
X, y = create_dataset(series, WINDOW)  # build dataset
N = X.shape[0]  # number of samples
train_N = int(N*0.8)  # 80% train split
X_train, y_train = X[:train_N], y[:train_N]  # training tensors
X_test, y_test = X[train_N:], y[train_N:]  # testing tensors
print(f'Train: {X_train.shape}, Test: {X_test.shape}')  # quick sanity check

In [ ]:
# Define a small LSTM regressor (sequence -> next value)
class LSTMRegressor(nn.Module):  # subclass nn.Module
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):  # config
        super().__init__()  # init parent
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)  # LSTM encoder
        self.fc = nn.Linear(hidden_size, 1)  # final linear layer to scalar
    def forward(self, x):  # forward pass
        out, _ = self.lstm(x)  # LSTM outputs for all time steps
        out = out[:, -1, :]  # take last time step representation
        return self.fc(out)  # map to prediction

model = LSTMRegressor().to(device)  # create model
criterion = nn.MSELoss()  # mean squared error loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)  # Adam optimizer

In [ ]:
# Train the model with mini-batches
EPOCHS = 60  # training epochs
batch_size = 64  # mini-batch size

for epoch in range(EPOCHS):  # loop over epochs
    model.train()  # training mode
    perm = torch.randperm(X_train.size(0))  # randomize order
    epoch_loss = 0.0  # track total loss
    for i in range(0, X_train.size(0), batch_size):  # iterate batches
        idx = perm[i:i+batch_size]  # batch indices
        xb = X_train[idx].to(device)  # batch inputs
        yb = y_train[idx].to(device)  # batch targets
        optimizer.zero_grad()  # reset gradients
        pred = model(xb)  # forward pass
        loss = criterion(pred, yb)  # compute MSE
        loss.backward()  # backprop gradients
        optimizer.step()  # update parameters
        epoch_loss += loss.item() * xb.size(0)  # accumulate
    if (epoch+1) % 10 == 0:  # log every 10 epochs
        print(f'Epoch {epoch+1}/{EPOCHS} - Train MSE: {epoch_loss/X_train.size(0):.4f}')  # progress

In [ ]:
# Evaluate on the test set and visualize
model.eval()  # eval mode
with torch.no_grad():  # disable gradients
    y_pred = model(X_test.to(device)).cpu().squeeze().numpy()  # predictions
    y_true = y_test.squeeze().numpy()  # ground truth

plt.figure(figsize=(12,4))  # create figure
plt.plot(y_true, label='True')  # plot truth
plt.plot(y_pred, label='Predicted')  # plot predictions
plt.title('LSTM Forecast: Next-step Prediction')  # title
plt.legend()  # legend
plt.tight_layout()  # layout
plt.show()  # render

### Key ideas
- Inputs shaped as (batch, seq_len, feature) with `batch_first=True`.
- This is a sequence-to-one setup: predict the next value from a recent window.
- Try GRU, multi-step forecasting, or add exogenous variables for richer models.